# 📷 Digital Image Enhancement: Section 2.1 Point Operations ($s = T(r)$)
**Course:** Image Processing and Computer Vision (Module 2)  
**Instructor:** Dr. Guindo (DAUST)  
**Topic:** Point Transformations ($s = T(r)$)

---
## 🎯 What is a Point Transformation? (Slide 11)
A **point operation** processes **one pixel at a time**:  
$$s = T(r)$$
- $r$ is the input intensity value at a pixel, $s$ is the output value.
- **Pixel coordinates and spatial neighbors are irrelevant.**
- Section 2.1 covers **seven core point transformations**:
  1. **Negative**: $s = 255 - r$
  2. **Linear**: $s = \alpha \cdot r + \beta$
  3. **Contrast stretch**: $s = 255 \frac{r - r_{\min}}{r_{\max} - r_{\min}}$ and Percentile Stretch
  4. **Log**: $s = c \cdot \log(1 + r)$
  5. **Gamma (Power Law)**: $s = 255 \cdot (r \div 255)^\gamma$
  6. **Threshold**: $s = 255$ if $r > T$, else $0$
  7. **Equalisation**: $s = \text{round}((L - 1) \cdot \text{cdf}(r))$
  8. **Bit-plane slicing** (Slide 26-27)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.degradation import (
    infect_low_contrast, infect_low_light, infect_overexposed
)
from src.point_transforms import (
    transform_negative, transform_linear, transform_minmax_stretch,
    transform_percentile_stretch, transform_log, transform_gamma,
    transform_threshold, extract_bit_planes
)
from src.histogram_ops import global_histogram_equalization
from src.utils import calculate_stats, calculate_psnr

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
print('Environment ready for Section 2.1 Point Operations!')

## 🖼️ Step 0: Load Original Reference Image

In [ ]:
orig_img = cv2.imread('original_image.png')
if orig_img is None:
    raise FileNotFoundError('original_image.png not found!')

orig_rgb = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
orig_stats = calculate_stats(orig_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(orig_rgb)
plt.title(f'Original Image Baseline\nMean: {orig_stats["mean"]}, Std: {orig_stats["std"]}')
plt.axis('off')

plt.subplot(1, 2, 2)
g_orig = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)
plt.hist(g_orig.ravel(), bins=256, range=(0, 256), color='navy', alpha=0.7)
plt.title(f'Intensity Histogram\nEntropy: {orig_stats["entropy"]} bits')
plt.xlabel('Intensity')
plt.ylabel('Pixel Count')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🦠 Step 1: Synthesizing Degradation States ("Infections")
We create three degraded image states to test our point transformations:
1. **Low Contrast Infection**: Dynamic range squashed into $[25, 75]$.
2. **Low Light Infection**: Brightness scaled down by $0.15$.
3. **Overexposed Infection**: Brightness shifted $+110$ with highlight clipping.

In [ ]:
inf_low_contrast = infect_low_contrast(orig_img, 25, 75)
inf_low_light = infect_low_light(orig_img, factor=0.15)
inf_overexposed = infect_overexposed(orig_img, shift=110, scale=1.2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
deg_list = [
    ("Low Contrast Infection", inf_low_contrast),
    ("Low Light Infection", inf_low_light),
    ("Overexposed Infection", inf_overexposed)
]

for ax, (title, img) in zip(axes, deg_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean:{st['mean']} Std:{st['std']}\nPSNR: {psnr_v} dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 1: Point Transformation Test Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔄 Step 2: Point Transforms 1 & 2 — Image Negative & Linear Transform (Slides 12-14)

* **1. Image Negative ($s = 255 - r$)** (Slide 12): Inverts black and white. Useful on medical mammograms/X-rays to reveal bright specks inside dark fields.
* **2. Linear Transform ($s = \alpha \cdot r + \beta$)** (Slide 13-14): $\alpha$ controls contrast (multiplies difference), $\beta$ controls brightness (shifts intensity).

In [ ]:
neg_img = transform_negative(orig_img)
linear_contrast = transform_linear(inf_low_contrast, alpha=1.8, beta=-40)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
step2_list = [
    ("Original Reference", orig_img),
    ("1. Negative s=255-r", neg_img),
    ("Low Contrast Input", inf_low_contrast),
    ("2. Linear s=1.8r-40", linear_contrast)
]

for ax, (title, img) in zip(axes, step2_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nStd:{st['std']} | PSNR:{psnr_v}dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 2: Negative and Linear Point Transformations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Step 3: Point Transform 3 — Contrast Stretching (Slides 15-19)

* **Ordinary Min-Max Contrast Stretching**: $s = 255 \cdot \frac{r - r_{\min}}{r_{\max} - r_{\min}}$
* **The One-Pixel Disaster (Slide 17)**: A single hot/dead sensor pixel disables ordinary min-max stretching.
* **Robust Percentile Stretching (Slide 18-19)**: Clips values between $1\%$ and $99\%$ percentiles ($lo, hi$) to make the stretch robust to outliers.

In [ ]:
minmax_stretched = transform_minmax_stretch(inf_low_contrast)
percentile_stretched = transform_percentile_stretch(inf_low_contrast, 1, 99)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
step3_list = [
    ("Low Contrast Input", inf_low_contrast),
    ("Min-Max Stretch", minmax_stretched),
    ("Percentile Stretch (1-99%)", percentile_stretched)
]

for idx, (title, img) in enumerate(step3_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    
    axes[0, idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, idx].set_title(f"{title}\nStd:{st['std']} | PSNR:{psnr_v}dB", fontsize=9)
    axes[0, idx].axis('off')
    
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[1, idx].hist(g.ravel(), bins=256, range=(0, 256), color='teal', alpha=0.7)
    axes[1, idx].set_xlim([0, 255])
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Step 3: Contrast Stretching Point Transformations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🌙 Step 4: Point Transforms 4 & 5 — Log Transform & Gamma Correction (Slides 20-24)

* **4. Log Transform**: $s = c \cdot \log(1 + r)$, expands low-intensity dark tones while compressing bright highlights.
* **5. Gamma Correction**: $s = 255 \cdot \left(\frac{r}{255}\right)^\gamma$
  * $\gamma < 1$ (e.g. $\gamma=0.4$): Brightens dark shadow detail.
  * $\gamma > 1$ (e.g. $\gamma=2.0$): Darkens over-exposed highlights.

In [ ]:
log_enhanced = transform_log(inf_low_light)
gamma_04 = transform_gamma(inf_low_light, gamma=0.4)
gamma_20 = transform_gamma(inf_overexposed, gamma=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
step4_list = [
    ("Low Light Input", inf_low_light),
    ("4. Log s=clog(1+r)", log_enhanced),
    ("5. Gamma (gamma=0.4)", gamma_04),
    ("Overexposed -> Gamma(gamma=2.0)", gamma_20)
]

for ax, (title, img) in zip(axes, step4_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean:{st['mean']} | PSNR:{psnr_v}dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 4: Non-Linear Point Transforms (Log & Gamma)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## ⚖️ Step 5: Point Transforms 6 & 7 — Thresholding & Histogram Equalization (Slides 25, 29-34)

* **6. Thresholding**: $s = 255$ if $r > T$, else $0$ (turns image into a binary mask).
* **7. Histogram Equalization**: $s = \text{round}((L - 1) \cdot \text{cdf}(r))$ (automatic global contrast enhancement using the image's own CDF).

In [ ]:
thresh_128 = transform_threshold(orig_img, T=128)
he_enhanced = global_histogram_equalization(inf_low_contrast)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
step5_list = [
    ("Original Reference", orig_img),
    ("6. Thresholding (T=128)", thresh_128),
    ("Low Contrast Input", inf_low_contrast),
    ("7. Histogram Equalization", he_enhanced)
]

for ax, (title, img) in zip(axes, step5_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nStd:{st['std']} | PSNR:{psnr_v}dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 5: Thresholding and Histogram Equalization Point Operations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔢 Step 6: Bit-Plane Slicing (Slides 26-27)

Decomposes every 8-bit pixel into 8 binary bit planes (Bits 7 to 0).

In [ ]:
bit_planes = extract_bit_planes(orig_img)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for idx, (b_name, b_img) in enumerate(bit_planes.items()):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(b_img, cmap='gray')
    axes[row, col].set_title(b_name, fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Step 6: Bit-Plane Slicing (Top Bits 7-5 carry 88% signal; Bits 2-0 carry noise)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏆 Quantitative Comparison Table for Section 2.1 Point Operations

In [ ]:
eval_list = [
    ("Original Reference", orig_img, orig_img),
    ("Infected: Low Contrast", inf_low_contrast, orig_img),
    ("Infected: Low Light", inf_low_light, orig_img),
    ("Infected: Overexposed", inf_overexposed, orig_img),
    ("1. Image Negative (s=255-r)", neg_img, orig_img),
    ("2. Linear Boost (a=1.8, b=-40)", linear_contrast, orig_img),
    ("3. Min-Max Contrast Stretch", minmax_stretched, orig_img),
    ("3. Percentile Stretch (1-99%)", percentile_stretched, orig_img),
    ("4. Log Transform s=clog(1+r)", log_enhanced, orig_img),
    ("5. Gamma Correction (gamma=0.4)", gamma_04, orig_img),
    ("5. Gamma Correction (gamma=2.0)", gamma_20, orig_img),
    ("6. Thresholding (T=128)", thresh_128, orig_img),
    ("7. Global Histogram Equalization", he_enhanced, orig_img),
]

print(f"{'Point Operation / Method':<32} | {'Mean':<6} | {'Std (Contrast)':<14} | {'Entropy':<8} | {'PSNR (dB)':<9}")
print("-" * 75)
for name, img, ref in eval_list:
    st = calculate_stats(img)
    psnr_val = calculate_psnr(ref, img)
    print(f"{name:<32} | {st['mean']:<6} | {st['std']:<14} | {st['entropy']:<8} | {psnr_val:<9}")